In [19]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb lightgbm

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


walmart-recruiting-store-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)
2026/07/09 14:52:16 INFO mlflow.store.db.utils: Updating database tables


In [20]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

In [21]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

((421570, 5), (8190, 12), (45, 3))

In [22]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [23]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
]

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        return df[FEATURE_COLS]

In [24]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('LightGBM_Training')

<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/2', creation_time=1783607827233, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783607827233, lifecycle_stage='active', name='LightGBM_Training', tags={}, trace_location=None, workspace='default'>

In [25]:
with mlflow.start_run(run_name='LightGBM_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

missing_markdown_pct,▁
negative_sales_rows,▁
missing_markdown_pct,0.55849
negative_sales_rows,1285


In [26]:
with mlflow.start_run(run_name='LightGBM_Feature_Selection'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_Feature_Selection', reinit='finish_previous')

    builder = FeatureBuilder(stores, features)
    X_all = builder.transform(train)

    probe = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, verbose=-1)
    probe.fit(X_all, y_train)

    importances = pd.Series(probe.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    mlflow.log_dict(importances.to_dict(), 'feature_importances.json')
    wandb.log({'feature_importances': wandb.Table(dataframe=importances.reset_index().rename(columns={'index': 'feature', 0: 'importance'}))})

    selected_features = importances[importances > 0].index.tolist()
    mlflow.log_param('n_features_selected', len(selected_features))
    mlflow.log_dict({'selected_features': selected_features}, 'selected_features.json')
    wandb.log({'n_features_selected': len(selected_features)})
    wandb.finish()

importances

n_features_selected,▁
n_features_selected,18


,0
Dept,3042
Size,843
Store,762
CPI,315
Week,312
Unemployment,162
Temperature,118
Type,96
Month,89
IsPreChristmas,63


In [27]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [28]:
params = dict(n_estimators=1000, num_leaves=127, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, verbose=-1)

with mlflow.start_run(run_name='LightGBM_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        train_mask = train['Date'] <= train_end
        val_mask = (train['Date'] > train_end) & (train['Date'] <= val_end)

        X_tr = builder.transform(train[train_mask])
        X_val = builder.transform(train[val_mask])
        y_tr = y_train[train_mask]
        y_val = y_train[val_mask]

        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        score = wmae(y_val.values, preds, train.loc[val_mask, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▃▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2265.82359
wmae_mean,3033.18764
wmae_std,715.75333


[np.float64(3988.550124880895),
 np.float64(2845.1892082804316),
 np.float64(2265.823587241299)]

In [29]:
with mlflow.start_run(run_name='LightGBM_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_Final', config=params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilder(stores, features)),
        ('model', LGBMRegressor(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/09 14:56:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,1682.18757


np.float64(1682.1875702102016)

In [30]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_LightGBM.ipynb in Colab"
    !git push

[main 9816cdb] Run model_experiment_LightGBM.ipynb in Colab
 8 files changed, 117 insertions(+)
 create mode 100644 mlruns/2/26a0f1b0ac954148b2680165452bf2c5/artifacts/feature_importances.json
 create mode 100644 mlruns/2/26a0f1b0ac954148b2680165452bf2c5/artifacts/selected_features.json
 create mode 100644 mlruns/2/models/m-1f2cbd75f35d44038f59b753127070e3/artifacts/MLmodel
 create mode 100644 mlruns/2/models/m-1f2cbd75f35d44038f59b753127070e3/artifacts/conda.yaml
 create mode 100644 mlruns/2/models/m-1f2cbd75f35d44038f59b753127070e3/artifacts/model.pkl
 create mode 100644 mlruns/2/models/m-1f2cbd75f35d44038f59b753127070e3/artifacts/python_env.yaml
 create mode 100644 mlruns/2/models/m-1f2cbd75f35d44038f59b753127070e3/artifacts/requirements.txt
Enumerating objects: 18, done.
Counting objects: 100% (18/18), done.
Delta compression using up to 2 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (13/13), 3.88 MiB | 3.00 MiB/s, done.
Total 13 (delta 2), reused 1 (delta